In [8]:
pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [16]:
import requests                     
from bs4 import BeautifulSoup        
import pandas as pd                 
import spacy                        
import nltk                        
from nltk.corpus import stopwords
import fitz  

nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marut\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\marut\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [78]:
import os
import re
import csv
import docx
from skill_parsing2 import extract_skills_from_resume, get_all_job_titles
from fuzzywuzzy import fuzz

def extract_text_from_docx(docx_path):
    doc = docx.Document(docx_path)
    text = "\n".join([para.text for para in doc.paragraphs])
    return text
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

def extract_name(text):
    lines = text.splitlines()

    for line in lines[:10]:
        line = line.strip()
        if not line:
            continue

        if any(char in line for char in "@/|•") or re.search(r'\d', line):
            continue

        if re.search(r"(analyst|engineer|manager|consultant|developer|nurse|student)", line.lower()):
            continue


        words = line.split()
        if len(words) >= 2 and all(word[0].isupper() for word in words if word.isalpha()):
            return line

    return ""

def extract_email(text):
    match = re.search(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", text)
    return match.group(0) if match else ""

def extract_phone(text):

    match = re.search(r"(?:\+1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}", text)
    return match.group(0) if match else ""
    
def extract_education(text):
    lines = text.splitlines()
    capture = False
    edu_lines = []
    edu_block = ""

    for line in lines:
        if "education" in line.lower():
            capture = True
            continue
        if capture:
            line = line.strip()
            if not line or re.match(r"^[A-Z\s]{5,}$", line):  # new section or all caps
                break
            edu_lines.append(line)
            edu_block += " " + line
            if len(edu_lines) >= 3:
                break

    match = re.search(r"(Bachelor|Master|PhD|Doctor|Diploma|Certificate)[^0-9\n]*(?:in)?[^0-9\n]*?\b(?:19|20)\d{2}\b", edu_block, re.IGNORECASE)
    if match:
        entry_text = match.group().strip()
        year_match = re.search(r"\b(19|20)\d{2}\b", entry_text)
        year = int(year_match.group()) if year_match else None

        parts = entry_text.split(str(year)) if year else [entry_text]
        course = parts[0].strip()
        institution = parts[1].strip() if len(parts) > 1 else ""
    else:
      
        if len(edu_lines) < 2:
            return []

        course = edu_lines[0]
        institution = edu_lines[1]
        year_line = edu_lines[2] if len(edu_lines) > 2 else ""
        year_match = re.search(r"\b(19|20)\d{2}\b", year_line or institution)
        year = int(year_match.group()) if year_match else None

  
    course_lower = course.lower()
    if "bachelor" in course_lower:
        degree_level = "Bachelor's"
    elif "master" in course_lower:
        degree_level = "Master's"
    elif "phd" in course_lower or "doctor" in course_lower:
        degree_level = "PhD"
    elif "diploma" in course_lower:
        degree_level = "Diploma"
    elif "certificate" in course_lower:
        degree_level = "Certificate"
    else:
        degree_level = "N/A"

    return [{
        "level": degree_level,
        "course": course.strip(),
        "institution": institution.strip(),
        "year": year
    }]
def extract_job_title(text):
   
    job_titles = get_all_job_titles()
    lines = text.splitlines()
    top_lines = [line.strip() for line in lines[:10] if line.strip()]
    best_score = 0
    best_title = ""
    for ref_title in job_titles:
        for line in top_lines:
            score = fuzz.token_sort_ratio(ref_title.lower(), line.lower())
            if score > best_score:
                best_score = score
                best_title = ref_title
    return best_title

def process_resume(resume_path, output_csv='parsed_resume.csv'):

    if resume_path.lower().endswith('.docx'):
        text = extract_text_from_docx(resume_path)
    elif resume_path.lower().endswith('.pdf'):
        text = extract_text_from_pdf(resume_path)
    else:
        print(f"Unsupported file format: {resume_path}")
        return

    name = extract_name(text)
    email = extract_email(text)
    phone = extract_phone(text)
    education = extract_education(text)
    job_title = extract_job_title(text)
    skills = extract_skills_from_resume(text, job_title)

    write_header = not os.path.isfile(output_csv) or os.path.getsize(output_csv) == 0

    with open(output_csv, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(['Name', 'Email', 'Phone', 'Education', 'Job Title', 'Skills'])

        if education:
            course = education[0]['course'] or ""
            institution = education[0]['institution'] or ""
            year = str(education[0]['year']) if education[0]['year'] else ""

            edu_parts = [course, institution, year]
            edu_str = ", ".join([part for part in edu_parts if part])
        else:
            edu_str = ""

        writer.writerow([
            name,
            email,
            phone,
            edu_str,
            job_title,
            ", ".join(skills)
        ])
    
    print(f"Appended parsed resume to {output_csv}")

if __name__ == "__main__":
    resume_path = r"C:\Users\marut\Desktop\capstone\sample.resume.docx"
    process_resume(resume_path)

Appended parsed resume to parsed_resume.csv


In [50]:
import importlib
import skill_parsing
importlib.reload(skill_parsing)

<module 'skill_parsing' from 'C:\\Users\\marut\\Capstone\\skill_parsing.py'>